# Vicsek Flock Lab: live demo for Part I (Anna Chen)

*Herding Randomness*, Applied Mathematics Senior Seminar.

Fourteen sheep in a 40 m pasture whose edges wrap around. Every second each sheep

1. **alignment**: takes the average heading of the neighbors within $R = 5$ m (itself included),
2. **random turn**: adds a random angle of up to $\pm\eta\pi$,
3. **constant speed**: walks 1 m in that direction.

Step 2 adds a dog (Jadhav et al. 2024 parameters, no noise) and a random target.

**How to run:** *Kernel → Restart & Run All*. Needs only `numpy` and `matplotlib`.
The six demo animations (sections 2 and 3) take about half a minute in total. Section 4 reproduces every
figure on the slides and takes a few minutes.

| section | what |
|---|---|
| 1 | the model: settings, Step 1 and Step 2 code (same lines as the *Pseudocode* slides), measurements |
| 2 | **demos 1–3**: Step 1 (no dog), $\eta$ = 0.1, 0.3, 0.6 |
| 3 | **demos 4–6**: Step 2 (one dog), $\eta$ = 0.1, 0.3, 0.6 |
| 4 | all slide figures: snapshots, RQ1, RQ2, RQ3, dog snapshot and violins |

## 1. The model
### Settings

In [ ]:
import numpy as np

N, R, V0, L = 14, 5.0, 1.0, 40.0        # sheep, neighbor radius (m), speed (m/s), pasture side (m)
ETA = 0.2                               # default noise
D_LINK, V_LINK, T_LINK = 5.0, 0.5, 5    # connecting-group thresholds: 5 m, 0.5 m/s, over 5 s

# dog and target (Jadhav et al. 2024 parameters)
R_D, RHO_D = 12.0, 1.0                  # sheep notice the dog within 12 m; weight of fleeing
D_REP = 2.0                             # "too close" distance (m)
F_N = D_REP * N ** (2 / 3)              # 11.6 m: farther than this from the center = straggler
P_D = D_REP * np.sqrt(N)                # 7.5 m: driving position behind the center
P_C = D_REP                             # 2 m: collecting position behind the straggler
V_DOG, CREEP = 1.5, 0.05                # dog speed, creep speed (m/s)
SUCCESS_R = 10.0                        # target reached: center within 10 m (and flock together)
COLLECT, DRIVE, CREEPING = 0, 1, 2

### Helpers
`shortest(d)`: the pasture wraps around, so a sheep near the right edge is close to one near the left edge.
The shortest separation is measured the short way, across the edge.

In [ ]:
def shortest(d):
    """Shortest separation across the edges of the pasture (it wraps around like a torus)."""
    return d - L * np.round(d / L)


def unit(v):
    n = np.hypot(v[0], v[1])
    return v / n if n > 0 else np.zeros(2)


def neighbors(x, i):
    """Indices of the sheep within R of sheep i (sheep i included)."""
    d = shortest(x - x[i])
    return np.flatnonzero((d ** 2).sum(1) <= R ** 2)


def new_flock(seed, eta=ETA, layout="cluster", a=4.0, b=4.0):
    """14 sheep in a tight 6 m x 6 m cluster in the middle (or an ellipse with semi-axes a, b)."""
    rng = np.random.default_rng(seed)
    c = L / 2
    if layout == "cluster":
        x = c + 6 * (rng.random((N, 2)) - 0.5)
    else:
        r, ph = np.sqrt(rng.random(N)), 2 * np.pi * rng.random(N)
        x = np.column_stack([c + a * r * np.cos(ph), c + b * r * np.sin(ph)])
    theta = 2 * np.pi * rng.random(N)
    return dict(x=x % L, X=x.copy(), theta=theta, hist=[x.copy()], eta=eta, rng=rng, t=0)


def remember(f, move):
    """Keep unwrapped positions of the last 30 s (for the recent velocity u_i)."""
    f["X"] = f["X"] + move
    f["hist"].append(f["X"].copy())
    if len(f["hist"]) > 31:
        f["hist"].pop(0)
    f["t"] += 1

### Step 1: move the flock (slide *Pseudocode*, left box)
```
for i in range(N):
    N_i     <- sheep within R
    theta_i <- atan2(sum sin theta_j, sum cos theta_j)        # alignment
    theta_i <- theta_i + eta*pi*Uniform(-1, 1)                # random turn
for i in range(N):
    x_i <- x_i + v0 (cos theta_i, sin theta_i)  mod L         # constant speed
```

In [ ]:
def move_flock(f):
    x, theta, eta, rng = f["x"], f["theta"], f["eta"], f["rng"]
    new_theta = np.empty(N)
    for i in range(N):
        Ni = neighbors(x, i)
        theta_bar = np.arctan2(np.sin(theta[Ni]).sum(), np.cos(theta[Ni]).sum())  # alignment
        new_theta[i] = theta_bar + eta * np.pi * rng.uniform(-1, 1)              # random turn
    move = np.zeros((N, 2))
    for i in range(N):
        move[i] = V0 * np.array([np.cos(new_theta[i]), np.sin(new_theta[i])])   # constant speed
    f["x"] = (x + move) % L
    f["theta"] = new_theta
    remember(f, move)

### Step 2: move the flock, with a dog (second *Pseudocode* slide)
```
for i in range(N):
    N_i <- sheep within R
    a_i <- unit(sum (cos theta_j, sin theta_j))               # alignment
    if |x_i - x_D| <= R_D:
        a_i <- a_i + rho_d unit(x_i - x_D)                    # flee the dog
    theta_i <- angle(a_i) + eta*pi*Uniform(-1, 1)             # random turn
move_dog()                                                    # collect or drive, no noise
for i in range(N):
    x_i <- x_i + v0 (cos theta_i, sin theta_i)  mod L         # constant speed
if flock reached target:
    target <- new random point
```

In [ ]:
def torus_mean(x):
    """Center of the flock on the wrapping pasture."""
    a = 2 * np.pi * x / L
    return (L * np.arctan2(np.sin(a).mean(0), np.cos(a).mean(0)) / (2 * np.pi)) % L


def new_target(rng, center):
    while True:                                           # random point at least 10 m away
        tgt = rng.random(2) * L
        if np.hypot(*shortest(tgt - center)) >= SUCCESS_R:
            return tgt


def new_herd(seed, eta=ETA):
    """Same starting flock as Step 1, plus a dog (placed as in Jadhav's code) and a target."""
    f = new_flock(seed, eta=eta)
    rng = f["rng"]
    corner = np.array([L / 2 + 3, L / 2 + 3])
    f["dog"] = (corner - 3 * R_D * rng.random(2)) % L
    a = 2 * np.pi * rng.random()
    f["dog_dir"] = np.array([np.cos(a), np.sin(a)])
    f["target"] = new_target(rng, torus_mean(f["x"]))
    f.update(reached=[], modes=[])
    return f


def move_dog(f):
    """Stroembom's collect-or-drive rule with Jadhav's parameters, no noise."""
    x, dog = f["x"], f["dog"]
    if np.linalg.norm(shortest(x - dog), axis=1).min() <= D_REP:      # a sheep within 2 m
        f["dog"] = (dog + CREEP * f["dog_dir"]) % L                     # creep
        return CREEPING
    center = torus_mean(x)
    rel = shortest(x - center)
    dist = np.linalg.norm(rel, axis=1)
    k = int(np.argmax(dist))
    if dist[k] >= F_N:                                                  # farthest sheep is a straggler
        goal, mode = center + (dist[k] + P_C) * rel[k] / dist[k], COLLECT   # go behind it
    else:
        b = shortest(center - f["target"])
        nb = np.hypot(*b)
        goal, mode = f["target"] + (nb + P_D) * b / max(nb, 1e-12), DRIVE  # go p_d behind center
    g = shortest(goal - dog)
    if np.hypot(*g) > 1e-12:
        f["dog_dir"] = g / np.hypot(*g)
    f["dog"] = (dog + V_DOG * f["dog_dir"]) % L                         # step at 1.5 m/s
    return mode


def move_flock_with_dog(f, flee=True):
    x, theta, eta, rng, dog = f["x"], f["theta"], f["eta"], f["rng"], f["dog"]
    new_theta = np.empty(N)
    for i in range(N):
        Ni = neighbors(x, i)
        a_i = unit(np.array([np.cos(theta[Ni]).sum(), np.sin(theta[Ni]).sum()]))  # alignment
        away = shortest(x[i] - dog)
        if flee and np.hypot(*away) <= R_D:
            a_i = a_i + RHO_D * unit(away)                                     # flee the dog
        new_theta[i] = np.arctan2(a_i[1], a_i[0]) + eta * np.pi * rng.uniform(-1, 1)  # random turn
    f["modes"].append(move_dog(f))
    move = np.zeros((N, 2))
    for i in range(N):
        move[i] = V0 * np.array([np.cos(new_theta[i]), np.sin(new_theta[i])])  # constant speed
    f["x"] = (x + move) % L
    f["theta"] = new_theta
    remember(f, move)
    center = torus_mean(f["x"])
    together = np.linalg.norm(shortest(f["x"] - center), axis=1).max() < F_N
    if np.hypot(*shortest(center - f["target"])) < SUCCESS_R and together:   # flock reached target
        f["reached"].append(f["t"])
        f["target"] = new_target(rng, center)                                  # new random point

### Measure the flock (slide *Pseudocode*, right box)
Two sheep are **linked** if they are within $D = 5$ m **and** their velocities over the last 5 s differ by at most
$V = 0.5$ m/s. A **group** is a chain of links, found by breadth-first search. For each group with at least 3 sheep,
PCA of the positions gives **roundness** $\lambda_2/\lambda_1$ and **sideways** $|\sin\alpha|$.

In [ ]:
def measure(f):
    x, theta = f["x"], f["theta"]
    h = f["hist"]
    k = max(0, len(h) - 1 - T_LINK)
    u = (h[-1] - h[k]) / max(1, len(h) - 1 - k)            # u_i = (x_i(t) - x_i(t-T)) / T
    d = shortest(x[None, :, :] - x[:, None, :])
    link = np.zeros((N, N), bool)
    for i in range(N):
        for j in range(N):
            if i != j and (d[i, j] ** 2).sum() <= D_LINK ** 2 and ((u[i] - u[j]) ** 2).sum() <= V_LINK ** 2:
                link[i, j] = True
    # groups = chains of links (breadth-first search: start from one sheep, add its linked
    # sheep, then theirs, until nothing new is added; positions are unwrapped across edges)
    label = -np.ones(N, int)
    px = np.zeros((N, 2))
    groups = []
    for s in range(N):
        if label[s] >= 0:
            continue
        label[s], px[s], members = len(groups), x[s], [s]
        for i in members:
            for j in np.flatnonzero(link[i] & (label < 0)):
                label[j], px[j] = len(groups), px[i] + d[i, j]
                members.append(j)
        groups.append(shape(np.array(members), px, theta))
    groups.sort(key=lambda g: -g["n"])
    P = float(np.hypot(np.cos(theta).mean(), np.sin(theta).mean()))
    return dict(P=P, groups=groups)


def shape(g, px, theta):
    """PCA of one group: roundness = lambda2/lambda1, sideways = |sin(long axis - heading)|."""
    p = px[g]
    c = p.mean(0)
    C = np.cov((p - c).T, bias=True) if len(g) > 1 else np.zeros((2, 2))
    lam, vec = np.linalg.eigh(C)                       # lam[1] = lambda_1 (larger)
    axis = np.arctan2(vec[1, 1], vec[0, 1])            # long axis
    hx, hy = np.cos(theta[g]).mean(), np.sin(theta[g]).mean()
    Pg, heading = np.hypot(hx, hy), np.arctan2(hy, hx)
    ok = len(g) >= 3 and lam[1] > 1e-9
    return dict(mem=g, n=len(g), c=c, l1=lam[1], l2=max(lam[0], 0.0), axis=axis, heading=heading,
                speed=V0 * Pg, spread=float(np.linalg.norm(p - c, axis=1).mean()),
                round=lam[0] / lam[1] if ok else np.nan,
                side=abs(np.sin(axis - heading)) if ok else np.nan)

### Drawing and animation

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, Circle
from IPython.display import HTML, display

COL = dict(sheep="#C2185B", dog="#1565C0", green="#2E7D32", orange="#EF6C00",
           gray="#616161", light="#BDBDBD", grass="#F4F7F1")
GROUP_COLS = [COL["sheep"], COL["green"], COL["orange"]]     # 1st, 2nd, 3rd largest group
plt.rcParams.update({"figure.dpi": 100, "font.size": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "mathtext.fontset": "cm",
                     "animation.embed_limit": 60})


def descriptors(m):
    """The 7 numbers we record per snapshot (largest group for the shape ones)."""
    b = m["groups"][0]
    return [m["P"], len(m["groups"]), b["n"], b["spread"], b["round"], b["side"], b["speed"]]


FEAT = ["Polarization P", "Groups", "Largest size", "Spread C", "Roundness", "Sideways",
        "Group velocity"]

In [ ]:
def draw_flock(ax, x, theta, m, f=None, title=""):
    """Sheep colored by connecting group, PCA ellipses; dog, its range and the target if f has them."""
    ax.clear()
    for rank, g in enumerate(m["groups"]):
        col = GROUP_COLS[rank] if rank < 3 and g["n"] > 1 else (COL["light"] if g["n"] > 1 else "white")
        p = x[g["mem"]]
        ax.quiver(p[:, 0], p[:, 1], np.cos(theta[g["mem"]]), np.sin(theta[g["mem"]]),
                  color=col if col != "white" else COL["gray"], angles="xy", scale_units="xy",
                  scale=1 / 1.6, width=0.006, zorder=3)
        ax.scatter(p[:, 0], p[:, 1], s=46, color=col, edgecolor="#212121", lw=0.8, zorder=4)
        if g["n"] >= 3:
            for ox in (-L, 0, L):                     # copies across the wrapping edges
                for oy in (-L, 0, L):
                    ax.add_patch(Ellipse(g["c"] % L + (ox, oy), 4 * np.sqrt(g["l1"]),
                                         4 * np.sqrt(g["l2"]) + 0.3, angle=np.degrees(g["axis"]),
                                         fill=False, ls="--", lw=1.6, color=col, zorder=2))
    if f is not None and "dog" in f:
        for ox in (-L, 0, L):
            for oy in (-L, 0, L):
                ax.add_patch(Circle(f["target"] + (ox, oy), SUCCESS_R, fill=False, ls="--", lw=1.6,
                                    color=COL["green"], zorder=1))
                ax.add_patch(Circle(f["dog"] + (ox, oy), R_D, fill=False, ls=":", lw=1.1,
                                    color=COL["dog"], zorder=1))
        ax.scatter(*f["target"], marker="*", s=220, color=COL["green"], zorder=5)
        ax.scatter(*f["dog"], marker="D", s=90, color=COL["dog"], zorder=6)
    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor(COL["grass"])
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_color(COL["gray"])
    ax.set_title(title, fontsize=12)


def legend_handles(dog):
    h = [Line2D([], [], marker="o", ls="", color=c, mec="#212121", label=l)
         for c, l in zip(GROUP_COLS + [COL["light"], "white"],
                         ["largest group", "2nd", "3rd", "smaller groups", "on its own"])]
    if dog:
        h += [Line2D([], [], marker="D", ls="", color=COL["dog"], label="dog (range 12 m)"),
              Line2D([], [], marker="*", ls="", ms=12, color=COL["green"], label="target")]
    return h

In [ ]:
def demo(step=1, eta=0.3, seconds=50, play_seconds=5, seed=(2026, 20, 0), substeps=2):
    """Run Step 1 (no dog) or Step 2 (dog) for `seconds` and return an animation that plays
    in about `play_seconds`. Every run with the same seed starts from the same flock."""
    f = new_flock(list(seed), eta=eta) if step == 1 else new_herd(list(seed), eta=eta)
    frames = []                                   # (x, theta, measurement, dog, target, t)
    m = measure(f)
    frames.append((f["x"].copy(), f["theta"].copy(), m, f.get("dog"), f.get("target"), 0.0))
    for t in range(seconds):
        x0, d0, tg0 = f["x"].copy(), f.get("dog"), f.get("target")
        if step == 1:
            move_flock(f)
        else:
            move_flock_with_dog(f)
        move = shortest(f["x"] - x0)
        dmove = shortest(f["dog"] - d0) if step == 2 else None
        m_new = measure(f)
        for s in range(1, substeps + 1):          # in-between frames for a smooth picture
            w = s / substeps
            xs = (x0 + w * move) % L
            ds = (d0 + w * dmove) % L if step == 2 else None
            mm = m_new if s == substeps else dict(m_new, groups=[dict(g, c=g["c"] - (1 - w) * move[g["mem"]].mean(0)) for g in m_new["groups"]])
            frames.append((xs, f["theta"].copy(), mm, ds, tg0 if s < substeps else f.get("target"),
                           t + w))
    fig, ax = plt.subplots(figsize=(5.6, 6.0))
    fig.legend(handles=legend_handles(step == 2), loc="lower center", ncol=3 if step == 1 else 4,
               frameon=False, fontsize=9)
    fig.subplots_adjust(left=0.03, right=0.97, top=0.92, bottom=0.12)
    name = "Step 1: no dog" if step == 1 else "Step 2: one dog"

    def show(k):
        x, th, mm, d, tg, t = frames[k]
        ff = dict(dog=d, target=tg) if step == 2 else None
        ng = len(mm["groups"])
        draw_flock(ax, x, th, mm, ff, title=f"{name},  $\\eta={eta}$,  t = {t:4.1f} s\n"
                   f"$P$ = {mm['P']:.2f},  {ng} group{'s' if ng > 1 else ''},  "
                   f"largest {mm['groups'][0]['n']} of 14")
        return []

    anim = animation.FuncAnimation(fig, show, frames=len(frames),
                                   interval=1000 * play_seconds / len(frames), blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode="once"))

## 2. Demos 1–3: Step 1, no dog
The same starting flock each time (a tight 6 m cluster); 50 s of simulation shown in about 5 s.
Colors = connecting groups (magenta = largest). Press ▶ to play; the − / + buttons change the speed.

**Demo 1: $\eta = 0.1$** (little noise)

In [ ]:
demo(step=1, eta=0.1)

**Demo 2: $\eta = 0.3$**

In [ ]:
demo(step=1, eta=0.3)

**Demo 3: $\eta = 0.6$** (a lot of noise)

In [ ]:
demo(step=1, eta=0.6)

## 3. Demos 4–6: Step 2, one dog
The same sheep and the same start, plus a dog (blue diamond; dotted circle = the 12 m within which sheep flee it)
and a target (green star; dashed circle = 10 m).

**Demo 4: $\eta = 0.1$**

In [ ]:
demo(step=2, eta=0.1)

**Demo 5: $\eta = 0.3$**

In [ ]:
demo(step=2, eta=0.3)

**Demo 6: $\eta = 0.6$**

In [ ]:
demo(step=2, eta=0.6)

*Try your own:* `demo(step=1 or 2, eta=..., seconds=..., seed=(any, numbers))`.

## 4. All slide figures
Same runs and seeds as the slides. Set `RUNS = 4` for a quick check (a few seconds per figure).

In [ ]:
RUNS, RUNS_A = 20, 300      # slides: 20 runs per eta, 300 runs per start shape (Test A)

### Demo slide: snapshots after 150 s

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.9))
for ax, eta in zip(axes, [0.1, 0.3, 0.6]):
    f = new_flock([7, int(eta * 10)], eta=eta)
    for _ in range(150):
        move_flock(f)
    m = measure(f)
    ng = len(m["groups"])
    draw_flock(ax, f["x"], f["theta"], m,
               title=f"$\\eta={eta}$:  $P={m['P']:.2f}$,  {ng} group{'s' if ng > 1 else ''},  "
                     f"largest {m['groups'][0]['n']} of 14")
fig.legend(handles=legend_handles(False) + [Line2D([], [], ls="--", color=COL["gray"],
           label="PCA ellipse (2 SD)")], loc="lower center", ncol=6, frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

### RQ1: noise $\to$ shape (20 runs × 400 s per $\eta$, skip the first 50 s, measure every 5 s)

In [ ]:
def sweep(runs=20, steps=400):
    """RQ1/RQ2: for each eta = 0, 0.1, ..., 1: `runs` runs x `steps` s from a tight cluster,
    skip the first 50 s, measure every 5 s."""
    rows = {}
    for eta in ETAS:
        rr = []
        for r in range(runs):
            f = new_flock([2026, int(eta * 100), r], eta=eta)
            for t in range(1, steps + 1):
                move_flock(f)
                if t > 50 and t % 5 == 0:
                    rr.append(descriptors(measure(f)))
        rows[eta] = np.array(rr, float)
    return rows


def q(a):
    a = a[np.isfinite(a)]
    return np.percentile(a, [25, 50, 75]) if a.size else np.full(3, np.nan)


ETAS = np.round(np.arange(0, 1.01, 0.1), 2)

In [ ]:
rows = sweep(runs=RUNS, steps=400)
panels = [(0, "Polarization $P$", (0, 1.02)), (4, "Roundness of largest group", (0, 1.02)),
          (1, "Number of groups", (0, 14.5)), (2, "Largest group (of 14)", (0, 14.5))]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (k, title, yl) in zip(axes, panels):
    st = np.array([q(rows[e][:, k]) for e in ETAS])
    ax.fill_between(ETAS, st[:, 0], st[:, 2], color=COL["dog"], alpha=0.18, lw=0)
    ax.plot(ETAS, st[:, 1], "o-", color=COL["dog"], ms=4.5)
    ax.axvline(ETA, color=COL["gray"], ls=":", lw=1.2)
    ax.set_title(title)
    ax.set_xlabel(r"noise $\eta$")
    ax.set_ylim(*yl)
    print(f"{title:28s} medians:", " ".join(f"{e:.1f}:{s:.2f}" for e, s in zip(ETAS, st[:, 1])))
fig.tight_layout()
plt.show()

### RQ2: what are the components that co-vary together? (PCA of 7 descriptors)

In [ ]:
Xall = np.vstack([np.column_stack([rows[e], np.full(len(rows[e]), e)]) for e in ETAS])
Xall = Xall[np.isfinite(Xall).all(1)]                      # roundness needs >= 3 sheep
Z = (Xall[:, :7] - Xall[:, :7].mean(0)) / Xall[:, :7].std(0)
lam, vec = np.linalg.eigh(np.cov(Z.T, bias=True))
lam, vec = lam[::-1], vec[:, ::-1]
if vec[0, 0] < 0:
    vec[:, 0] *= -1                                         # PC1 toward higher polarization
if vec[4, 1] < 0:
    vec[:, 1] *= -1                                         # PC2 toward higher roundness
ev = lam / lam.sum()
S = Z @ vec[:, :2]
fig, (a2, a1) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw=dict(width_ratios=[1, 1.25]))  # loadings left
idx = np.random.default_rng(0).permutation(len(S))[:4000]
sc = a1.scatter(S[idx, 0], S[idx, 1], c=Xall[idx, 7], cmap="viridis", s=6, alpha=0.6, lw=0)
fig.colorbar(sc, ax=a1, pad=0.01).set_label(r"noise $\eta$")
a1.set_xlabel(f"PC1 ({100 * ev[0]:.0f}% of variance)")
a1.set_ylabel(f"PC2 ({100 * ev[1]:.0f}%)")
a1.set_title("each dot = one snapshot of the flock")
yy = np.arange(7)
a2.barh(yy + 0.2, vec[:, 0], 0.38, color=COL["dog"], label="PC1")
a2.barh(yy - 0.2, vec[:, 1], 0.38, color=COL["sheep"], label="PC2")
a2.set_yticks(yy, FEAT)
a2.invert_yaxis()
a2.axvline(0, color=COL["gray"], lw=1)
a2.set_xlabel("loading")
a2.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
fig.tight_layout()
plt.show()
print(f"PC1 explains {ev[0]:.0%}, PC2 {ev[1]:.0%} of the variance ({len(S)} snapshots)")

### RQ3: does the initial flock shape change its speed? (Test A)
Start the same sheep as a disk or as a long ellipse (same area, same random headings) and watch the flock's velocity.

In [ ]:
def test_a(shape_, r):
    """Start the flock as a disk or an ellipse of the same area, record its velocity v0*P."""
    a, b = (4.0, 4.0) if shape_ == "round" else (8.0, 2.0)
    f = new_flock([5003, r], layout="ellipse", a=a, b=b)
    v = [V0 * np.hypot(np.cos(f["theta"]).mean(), np.sin(f["theta"]).mean())]
    for _ in range(40):
        move_flock(f)
        v.append(V0 * np.hypot(np.cos(f["theta"]).mean(), np.sin(f["theta"]).mean()))
    return v


curves = {s: np.mean([test_a(s, r) for r in range(RUNS_A)], axis=0) for s in ("round", "elongated")}
fig, a1 = plt.subplots(figsize=(7, 4.2))
for s, col, lab in (("round", COL["dog"], "round start (disk, radius 4 m)"),
                    ("elongated", COL["orange"], "elongated start (16 m x 4 m)")):
    a1.plot(np.arange(41), curves[s], color=col, lw=2, label=lab)
    a1.plot(10, curves[s][10], "o", color=col, ms=6)
    a1.annotate(f"{curves[s][10]:.2f}", (10, curves[s][10]), textcoords="offset points",
                xytext=(6, 4 if s == "round" else -14), color=col)
a1.axvline(10, color=COL["gray"], ls=":", lw=1.2)
a1.set_ylim(0, 1.02)
a1.set_xlabel("seconds after release")
a1.set_ylabel("flock velocity $v_0 P$ (m/s)")
a1.set_title(f"Test A: set the start shape ({RUNS_A} runs each)")
a1.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()
print(f"after 10 s: round {curves['round'][10]:.3f} m/s, elongated {curves['elongated'][10]:.3f} m/s")
print(f"after 40 s: round {curves['round'][40]:.3f} m/s, elongated {curves['elongated'][40]:.3f} m/s")

### Step 2: one dog vs no dog (same 20 starting flocks, $\eta = 0.2$)

In [ ]:
def herd_run(r, steps=400, flee=True):
    f = new_herd([2026, 20, r])                     # same starting flock as no-dog run r
    rr = []
    for t in range(1, steps + 1):
        move_flock_with_dog(f, flee=flee)
        if t > 50 and t % 5 == 0:
            rr.append(descriptors(measure(f)))
    modes = np.array(f["modes"][50:])
    return rr, len(f["reached"]), [(modes == k).mean() for k in (COLLECT, DRIVE, CREEPING)]


def vicsek_run(r, steps=400):
    f = new_flock([2026, 20, r])
    rr = []
    for t in range(1, steps + 1):
        move_flock(f)
        if t > 50 and t % 5 == 0:
            rr.append(descriptors(measure(f)))
    return rr


vic = np.array([row for r in range(RUNS) for row in vicsek_run(r)], float)
dog_runs = [herd_run(r) for r in range(RUNS)]
blind_runs = [herd_run(r, flee=False) for r in range(RUNS)]      # sheep ignore the dog
H = np.array([row for rr, _, _ in dog_runs for row in rr], float)

# one moment of the dog model: run 0 at t = 150 s
f = new_herd([2026, 20, 0])
for _ in range(150):
    move_flock_with_dog(f)
m = measure(f)
fig = plt.figure(figsize=(15, 4.8))
ax0 = fig.add_axes([0.0, 0.12, 0.27, 0.8])
draw_flock(ax0, f["x"], f["theta"], m, f, title=f"$t=150$ s: {len(m['groups'])} groups")
ax0.legend(handles=legend_handles(True)[:1] + legend_handles(True)[-2:], loc="upper center",
           bbox_to_anchor=(0.5, -0.01), ncol=3, frameon=False, fontsize=9)
axes = [fig.add_axes([0.33 + 0.23 * i, 0.12, 0.19, 0.78]) for i in range(3)]
for a, (j, title, yl) in zip(axes, [(1, "number of groups", (0, 14.5)), (4, "roundness", (0, 1.0)),
                                     (5, "sideways", (0, 1.0))]):
    data = [d[np.isfinite(d)] for d in (vic[:, j], H[:, j])]
    parts = a.violinplot(data, positions=[0, 1], widths=0.75, showmedians=True, showextrema=False)
    for pc, col in zip(parts["bodies"], [COL["gray"], COL["dog"]]):
        pc.set_facecolor(col)
        pc.set_alpha(0.35)
    parts["cmedians"].set_color("#212121")
    a.set_xticks([0, 1], ["no dog", "one dog"])
    a.set_ylim(*yl)
    a.set_title(title)
    for x0, d in zip([0, 1], data):
        md = np.median(d)
        a.text(x0 + 0.08, md, f"{md:.0f}" if j == 1 else f"{md:.2f}",
               va="top" if md > 0.9 * yl[1] else "bottom")
plt.show()
frac = np.mean([fr for _, _, fr in dog_runs], axis=0)
print(f"{RUNS} runs x 350 s analyzed = {RUNS * 350} s, {len(H)} snapshots per violin")
print(f"dog's time: collect {frac[0]:.0%}, drive {frac[1]:.0%}, creep {frac[2]:.1%}")
print(f"targets reached per run: {np.mean([n for _, n, _ in dog_runs]):.2f} "
      f"(sheep flee the dog) vs {np.mean([n for _, n, _ in blind_runs]):.2f} (sheep ignore the dog)")